RECCOMENDATION APP

In [147]:
import pandas as pd
from data_load import df

In [148]:
# Filtrujeme pouze explicitní hodnocení
df = df[df["Book-Rating"] > 0]
print(f"Explicit ratings: {len(df):,} ({len(df)/len(df)*100:.1f}%)")

Explicit ratings: 383,842 (100.0%)


1. IDENTIFIKACE FANBASE

In [149]:
INPUT_BOOK = "Twilight"
FAN_THRESHOLD = 8

# Všechna ISBN odpovídající vstupní knize
input_book_isbns = df[
    df["Book-Title"].str.contains(INPUT_BOOK, na=False, case=False)
]["ISBN"].unique()

print(f"Nalezeno {len(input_book_isbns)} ISBN pro '{INPUT_BOOK}'")

# uživatele s ratingem >= FAN_THRESHOLD pro alespoň jedno z těchto ISBN
fans = df[
    (df["ISBN"].isin(input_book_isbns)) & 
    (df["Book-Rating"] >= FAN_THRESHOLD)
]["User-ID"].unique()

print(f"Nalezeno {len(fans)} fanoušků knihy '{INPUT_BOOK}'")

Nalezeno 103 ISBN pro 'Twilight'
Nalezeno 182 fanoušků knihy 'Twilight'


2. OTHER FANBASE BOOKS

In [150]:
# Co dalšího fanoušci hodnotili

fan_ratings = df[
    (df["User-ID"].isin(fans)) & 
    (~df["ISBN"].isin(input_book_isbns))
]

print(f"Počet hodnocení od fanoušků: {len(fan_ratings):,}")
print(f"Počet unikátních knih, které hodnotili: {fan_ratings['ISBN'].nunique():,}")

Počet hodnocení od fanoušků: 31,911
Počet unikátních knih, které hodnotili: 25,719


3. BOOK STATISTICS

In [151]:
# Knižní statistiky

book_stats = fan_ratings.groupby(["ISBN", "Book-Title", "Book-Author"]).agg(
    fan_count=("Book-Rating", "size"),
    avg_rating=("Book-Rating", "mean"),
).reset_index()

print(f"Počet unikátních knih: {len(book_stats):,}")
print(book_stats.head())

Počet unikátních knih: 25,719
         ISBN                                         Book-Title  \
0  0000913154  The Way Things Work: An Illustrated Encycloped...   
1  0001944711                    Count Duckula: Vampire Vacation   
2  0002005018                                       Clara Callan   
3  0002118580                                Audacity to believe   
4  0002158973                               Landscape and Memory   

                     Book-Author  fan_count  avg_rating  
0  C. van Amerongen (translator)          1         8.0  
1               Maureen Spurgeon          1         6.0  
2           Richard Bruce Wright          1         8.0  
3                 Sheila Cassidy          1         9.0  
4                   Simon Schama          1         8.0  


4. NUMBER OF RATINGS THRESHOLD

In [152]:
# Filtr na knihy s dostatečným signálem

MIN_FANS = 10

book_stats_filtered = book_stats[book_stats["fan_count"] >= MIN_FANS]

print(f"Knih po filtru (fan_count >= {MIN_FANS}): {len(book_stats_filtered):,}")

Knih po filtru (fan_count >= 10): 15


5. TOP 10 BOOKS

In [153]:
TOP_N = 10

recommendations = book_stats_filtered.sort_values(
    "avg_rating", ascending=False
).head(TOP_N)

recommendations[["Book-Title", "Book-Author", "avg_rating", "fan_count"]]

,Book-Title,Book-Author,avg_rating,fan_count
14589,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,9.642857,14
9655,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,9.615385,13
9615,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,9.500000,10
9588,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,9.000000,11
8098,The Da Vinci Code,Dan Brown,8.928571,14
4656,A Kiss of Shadows (Meredith Gentry Novels (Pap...,LAURELL K. HAMILTON,8.642857,14
3920,The Lovely Bones: A Novel,Alice Sebold,8.600000,10
12982,Narcissus in Chains (Anita Blake Vampire Hunte...,Laurell K. Hamilton,8.500000,10
9046,The Bad Place,Dean R. Koontz,8.500000,10
4452,Jurassic Park,Michael Crichton,8.307692,13
